<a href="https://colab.research.google.com/github/ADRIANVM117/data-science-portfolio/blob/main/scenario-portfolio-lab/notebooks/01_data_foundation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

El botón abre esta notebook en Google Colab. Allí ejecuta primero la siguiente celda de preparación para obtener el código y los reportes públicos; no descarga datos de mercado ni solicita credenciales.

In [1]:
# Colab-only setup. Local execution deliberately performs no clone or installation.
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore  # available only in Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PUBLIC_REPOSITORY = 'https://github.com/ADRIANVM117/data-science-portfolio.git'
    REPOSITORY_ROOT = Path('/content/data-science-portfolio')
    PROJECT_ROOT = REPOSITORY_ROOT / 'scenario-portfolio-lab'
    if not REPOSITORY_ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', PUBLIC_REPOSITORY, str(REPOSITORY_ROOT)], check=True)
    if not ((PROJECT_ROOT / 'config' / 'universe.csv').is_file() and (PROJECT_ROOT / 'src' / 'scenario_portfolio_lab').is_dir()):
        raise RuntimeError('The public repository clone is incomplete or has an unexpected layout.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(PROJECT_ROOT / 'requirements.txt')], check=True)
    os.chdir(PROJECT_ROOT)
    print(f'Colab preparation complete: {PROJECT_ROOT}')
else:
    print('Local execution detected: no clone and no installation were performed.')


Local execution detected: no clone and no installation were performed.


# 01 — Fundamentos de datos para investigación de portafolios

**Pregunta guía:** ¿qué debe cumplir una serie histórica antes de entrar en un modelo de portafolios?

Esta notebook explica y audita la frontera de datos del proyecto. No descarga datos, no requiere credenciales y no construye escenarios, optimización ni recomendaciones.

Etiquetas de evidencia usadas aquí:

- **Comprobado mediante ejecución en la notebook:** ejemplos sintéticos y lectura/validación de los reportes públicos.
- **Respaldado por reportes del corte:** conteos y diagnósticos de la ejecución sellada del 11 de octubre de 2026.
- **Comprobable únicamente con artefactos locales:** matrices y raw ignorados por Git; su inspección queda desactivada por defecto.

In [2]:
from __future__ import annotations

import csv
import json
import sys
from datetime import date
from pathlib import Path

import polars as pl


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if ((candidate / 'config' / 'universe.csv').is_file()
                and (candidate / 'src' / 'scenario_portfolio_lab').is_dir()):
            return candidate
    raise RuntimeError(
        'Repository files config/universe.csv and src/scenario_portfolio_lab were not found. '
        'Run this notebook from within the complete scenario-portfolio-lab repository.'
    )


ROOT = find_project_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
DOCS = ROOT / 'docs'
FINAL_SNAPSHOT = DOCS / 'universe_snapshots' / 'entry_2026-10-11_cut_2026-10-09_full_window_final'
INSPECT_LOCAL_MATRICES = False  # Optional, ignored local artifacts only; never needed below.
print('Public mode: no API calls, no credentials, no local matrices required.')


Public mode: no API calls, no credentials, no local matrices required.


## A. Problema y contrato

La observación no comienza con una matriz: comienza con un contrato. Para este corte la ventana es **2018-01-02 a 2026-10-09**, la frecuencia es diaria y la moneda de cotización y análisis es USD. El catálogo es contemporáneo; no debe presentarse como un universo punto-en-el-tiempo.

No se imputan precios ni retornos. La ausencia de una observación se conserva para poder decidir si bloquea admisión. También hay tres conjuntos distintos: el catálogo inicial, el universo de entrada capturado y el universo elegible de datos. Este último no es una selección de cartera.

In [3]:
manifest = json.loads((FINAL_SNAPSHOT / 'entry_universe_manifest.json').read_text(encoding='utf-8'))
counts = manifest['counts']
assert counts['included'] + counts['structural_exclusions'] + counts['out_of_scope_pending'] == 202
contract = pl.DataFrame({
    'property': ['Window', 'Frequency', 'Quotation and analysis currency', 'Imputation', 'Catalog interpretation'],
    'value': ['2018-01-02 to 2026-10-09', 'Daily', 'USD', 'None', 'Contemporary; not point-in-time'],
})
print('COMPROBADO MEDIANTE EJECUCIÓN: sealed-manifest arithmetic reconciles to 202 records.')
contract


COMPROBADO MEDIANTE EJECUCIÓN: sealed-manifest arithmetic reconciles to 202 records.


property,value
str,str
"""Window""","""2018-01-02 to 2026-10-09"""
"""Frequency""","""Daily"""
"""Quotation and analysis currenc…","""USD"""
"""Imputation""","""None"""
"""Catalog interpretation""","""Contemporary; not point-in-tim…"


## B. Identidad e ingesta

Un ticker de referencia no identifica por sí solo el instrumento que responde un proveedor. Antes de admitir una serie se distingue identidad, símbolo del proveedor, mercado, moneda, clase de activo y, para ETFs, estructura directa/no apalancada. Una coincidencia de búsqueda no prueba todo lo anterior.

**Raw** conserva los bytes originales; los hashes y manifiestos enlazan esos bytes con el corte y la procedencia. **Silver** hace explícito el esquema sin reemplazar OHLC reportado por `adjusted_close`. El ejemplo siguiente es sintético y no representa una captura real.

In [4]:
from scenario_portfolio_lab.ingest import parse_daily_adjusted

synthetic_response = {
    'Meta Data': {'2. Symbol': 'DEMO', '3. Last Refreshed': '2018-01-03', '5. Time Zone': 'US/Eastern'},
    'Time Series (Daily)': {
        '2018-01-03': {'1. open': '10', '2. high': '11', '3. low': '9.5', '4. close': '10.5', '5. adjusted close': '10.4', '6. volume': '100', '7. dividend amount': '0.0', '8. split coefficient': '1.0'},
        '2018-01-02': {'1. open': '9.5', '2. high': '10.2', '3. low': '9.2', '4. close': '10.0', '5. adjusted close': '10.0', '6. volume': '110', '7. dividend amount': '0.0', '8. split coefficient': '1.0'},
    },
}
silver_demo = parse_daily_adjusted(synthetic_response, 'DEMO')
print('COMPROBADO MEDIANTE EJECUCIÓN: synthetic provider response parsed into Silver schema.')
silver_demo.select('session_date', 'open', 'close', 'adjusted_close', 'volume', 'dividend_amount', 'split_coefficient')


COMPROBADO MEDIANTE EJECUCIÓN: synthetic provider response parsed into Silver schema.


session_date,open,close,adjusted_close,volume,dividend_amount,split_coefficient
date,f64,f64,f64,i64,f64,f64
2018-01-02,9.5,10.0,10.0,110,0.0,1.0
2018-01-03,10.0,10.5,10.4,100,0.0,1.0


## C. Calidad y cobertura

Gold comprueba que Silver tiene esquema válido, fechas únicas, precios finitos y económicamente válidos, campos corporativos reportados y un precio al corte. **Elegibilidad** añade identidad/mercado/moneda/estructura permitida y cobertura completa de la ventana común. Por eso una serie puede ser Gold-approved sin entrar en la matriz elegible.

El reporte final documenta que FUBO tiene cinco faltantes iniciales y dos interiores. No se atribuyen automáticamente los cinco iniciales a una IPO: ese corte no añadió evidencia fechada de listado. La corrección fue hacer que `leading_nulls` también bloqueara cobertura completa; el resultado cambió de 137 a 125.

In [5]:
coverage_path = FINAL_SNAPSHOT / 'session_observation_report.csv'
with coverage_path.open(encoding='utf-8', newline='') as handle:
    coverage_rows = list(csv.DictReader(handle))
by_symbol = {row['symbol']: row for row in coverage_rows}
assert len(coverage_rows) == 138
assert int(by_symbol['AAPL']['missing_expected_sessions']) == 0
assert int(by_symbol['ABNB']['missing_expected_sessions']) == 741
assert int(by_symbol['FUBO']['missing_expected_sessions']) == 7
coverage_summary = json.loads((DOCS / 'full_window_coverage_summary_2026-10-11.json').read_text(encoding='utf-8'))
assert coverage_summary['counts']['captured_instruments'] == len(coverage_rows) == 138
assert coverage_summary['counts']['eligible_complete_window'] == 125
assert coverage_summary['counts']['blocked_by_coverage'] == 13
fubo_coverage = coverage_summary['instrument_evidence']['FUBO']
assert (fubo_coverage['leading_nulls'], fubo_coverage['interior_nulls'], fubo_coverage['trailing_nulls']) == (5, 2, 0)
comparison = pl.DataFrame([
    {'instrument': 'AAPL', 'case': 'complete coverage example', **{k: by_symbol['AAPL'][k] for k in ('expected_sessions', 'observed_sessions', 'missing_expected_sessions')}},
    {'instrument': 'ABNB', 'case': 'insufficient history example', **{k: by_symbol['ABNB'][k] for k in ('expected_sessions', 'observed_sessions', 'missing_expected_sessions')}},
    {'instrument': 'FUBO', 'case': f"{fubo_coverage['leading_nulls']} leading + {fubo_coverage['interior_nulls']} interior gaps", **{k: by_symbol['FUBO'][k] for k in ('expected_sessions', 'observed_sessions', 'missing_expected_sessions')}},
])
print('COMPROBADO MEDIANTE EJECUCIÓN: final public report has 138 rows and the stated AAPL/ABNB/FUBO counts.')
print('COMPROBADO MEDIANTE EJECUCIÓN: structured FUBO evidence records 5 leading and 2 interior gaps.')
print('RESPALDADO POR REPORTE DEL CORTE: the 137 -> 125 correction rationale.')
comparison


COMPROBADO MEDIANTE EJECUCIÓN: final public report has 138 rows and the stated AAPL/ABNB/FUBO counts.
COMPROBADO MEDIANTE EJECUCIÓN: structured FUBO evidence records 5 leading and 2 interior gaps.
RESPALDADO POR REPORTE DEL CORTE: the 137 -> 125 correction rationale.


instrument,case,expected_sessions,observed_sessions,missing_expected_sessions
str,str,str,str,str
"""AAPL""","""complete coverage example""","""2205""","""2205""","""0"""
"""ABNB""","""insufficient history example""","""2205""","""1464""","""741"""
"""FUBO""","""5 leading + 2 interior gaps""","""2205""","""2198""","""7"""


## D. Precios y retornos

OHLC es lo reportado por el proveedor; `adjusted_close` se conserva aparte y se usa para retornos simples bajo la convención del proveedor. Un retorno diario es `adjusted_close[t] / adjusted_close[t-1] - 1` únicamente cuando ambas fechas son sesiones esperadas consecutivas. El primer retorno queda nulo porque no hay un precio previo aprobado.

El siguiente ejemplo tiene una sesión esperada faltante. Aunque hay precios al inicio y al final, el cambio no se etiqueta como retorno diario.

### Ejemplos sintéticos de acciones corporativas y alineación

- **Split 2:1:** un precio de 100 que pasa a 50 después de un split no implica una pérdida de valor de la posición: dos acciones de 50 siguen representando 100.
- **Dividendo:** con precio inicial 100, precio final 99 y efectivo distribuido de 2, el retorno total es `(99 + 2) / 100 - 1 = 1%`.
- Si `adjusted_close` ya incorpora el efecto del dividendo bajo la convención del proveedor, sumar de nuevo el efectivo cuenta el dividendo dos veces.
- Ordenar columnas de instrumentos de manera independiente destruye la coincidencia histórica entre fechas y activos; la alineación siempre debe conservar la misma clave temporal.

Son ejemplos sintéticos para razonar sobre convenciones. No verifican ni validan los ajustes corporativos reales del proveedor.

In [6]:
from scenario_portfolio_lab.matrices import simple_returns

expected = {date(2018, 1, 2), date(2018, 1, 3), date(2018, 1, 4)}
prices_with_gap = pl.DataFrame({
    'session_date': [date(2018, 1, 2), date(2018, 1, 4)],
    'DEMO': [100.0, 110.0],
})
returns_with_gap = simple_returns(prices_with_gap, {'DEMO': expected})
assert returns_with_gap['DEMO'].to_list() == [None, None]
print('COMPROBADO MEDIANTE EJECUCIÓN: the two-session price change is not labelled a daily return.')
returns_with_gap


COMPROBADO MEDIANTE EJECUCIÓN: the two-session price change is not labelled a daily return.


session_date,DEMO
date,f64
2018-01-02,null
2018-01-04,null


### Ejercicios manuales — dejar sin resolver

1. Si las sesiones esperadas fueran martes, miércoles y jueves, ¿qué información adicional necesitarías para decidir si un precio observado el jueves puede producir un retorno diario?
2. ¿Cuándo sería apropiado definir otra ventana común en vez de relajar una regla de cobertura?
3. ¿Qué parte de la política de `adjusted_close` deberías contrastar con documentación del proveedor antes de interpretar retornos como rendimiento económico?

No hay solución programada en esta notebook: las preguntas quedan para trabajarse con el usuario y no bloquean la ejecución.

## E. Resultado y límites

La conciliación del catálogo es 202 = 138 incluidos + 14 excluidos estructuralmente + 50 pendientes fuera del alcance. De los 138 capturados, 125 pasan la elegibilidad de datos de ventana completa y 13 quedan bloqueados por cobertura.

La matriz elegible tiene 2,205 fechas de precios y una columna de fecha más 125 instrumentos. Cada instrumento tiene 2,204 retornos utilizables: el único nulo admitido es la primera sesión, sin precio previo. Una matriz completa no demuestra estacionariedad, validez predictiva ni alpha; sólo fija un punto de partida controlado para estudiar después escenarios y probabilidades.

In [7]:
assert counts == {'equities': 100, 'etfs': 38, 'included': 138, 'out_of_scope_pending': 50, 'structural_exclusions': 14}
matrix_summary = coverage_summary['eligible_matrix']
assert (matrix_summary['price_dates'], matrix_summary['instrument_columns']) == (2205, 125)
assert (matrix_summary['usable_returns_per_instrument'], matrix_summary['initial_nulls_per_instrument'], matrix_summary['subsequent_return_nulls_per_instrument']) == (2204, 1, 0)
summary = pl.DataFrame({
    'stage': ['Original catalog', 'Approved and captured input', 'Data-eligible complete window', 'Eligible price matrix', 'Usable returns per eligible instrument'],
    'result': ['202 records', '138 instruments', '125 instruments; 13 coverage blocks', '2,205 dates × (date + 125 instruments)', '2,204; first return intentionally null'],
})
print('COMPROBADO MEDIANTE EJECUCIÓN: public manifests and reports reconcile these final counts.')
summary


COMPROBADO MEDIANTE EJECUCIÓN: public manifests and reports reconcile these final counts.


stage,result
str,str
"""Original catalog""","""202 records"""
"""Approved and captured input""","""138 instruments"""
"""Data-eligible complete window""","""125 instruments; 13 coverage b…"
"""Eligible price matrix""","""2,205 dates × (date + 125 inst…"
"""Usable returns per eligible in…","""2,204; first return intentiona…"


### Inspección opcional de artefactos locales

`INSPECT_LOCAL_MATRICES` permanece en `False`. Activarlo requeriría matrices locales ignoradas por Git y permite comprobar directamente dimensiones, nulos y fechas. La auditoría pública de esta notebook **no** equivale a verificar esos archivos: los conteos anteriores están respaldados por reportes del corte.

In [8]:
if INSPECT_LOCAL_MATRICES:
    local_run = ROOT / 'data' / 'published' / 'cut_2026-10-09_run_2026-10-11_full_window_final'
    local_matrix = local_run / 'gold' / 'eligible_simple_total_return_matrix.parquet'
    local_universe = local_run / 'reports' / 'eligible_universe.csv'
    if not local_matrix.is_file() or not local_universe.is_file():
        raise FileNotFoundError('Optional ignored matrix or eligible-universe report is unavailable locally.')
    inspected = pl.read_parquet(local_matrix)
    assert inspected.columns[0] == 'session_date'
    assert inspected['session_date'].is_sorted() and inspected['session_date'].n_unique() == inspected.height
    symbols = inspected.columns[1:]
    assert len(symbols) == 125 and inspected.width == 126
    assert all(value == 1 for value in inspected.select(pl.exclude('session_date').null_count()).row(0))
    assert all(value == 0 for value in inspected.slice(1).select(pl.exclude('session_date').null_count()).row(0))
    assert not any(inspected.select(pl.exclude('session_date').is_nan().any()).row(0))
    assert not any(inspected.select(pl.exclude('session_date').is_infinite().any()).row(0))
    eligible_report = pl.read_csv(local_universe).filter(pl.col('universe_status') == 'ELIGIBLE')
    assert set(symbols) == set(eligible_report['symbol'].to_list())
    print({'shape': inspected.shape, 'validated': 'dates, symbols, first-null-only, finite returns'})
else:
    print('Optional local-matrix inspection remains disabled.')


Optional local-matrix inspection remains disabled.
